In [7]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [8]:
dataset = pd.read_parquet('train-00000-of-00001-1ae224438dce829b.parquet')
dataset
results_df = pd.DataFrame(columns=["bleu_score","instruction", "reference", "generated"])

In [9]:
def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    return model, tokenizer

In [10]:
def generate_response(model, tokenizer, prompt, max_length=256):
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Initialize the progress bar
    total_steps = max_length  # Number of decoding steps corresponds to max_length
    with tqdm(total=total_steps, desc="Generating response", unit="step") as pbar:
        # Generate the response
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True,
            # Update the progress bar on each decoding step
            output_scores=True, 
            return_dict_in_generate=True,
        )
        
        # Update the progress bar
        for _ in range(len(outputs.sequences[0])):
            pbar.update(1)
    
    # Decode the generated response
    response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return response

In [11]:
def evaluate_model(dataset, model, tokenizer):
    global results_df
    bleu_scores = []
    for idx in range(len(dataset)):
        instruction = dataset.at[idx, "instruction"]
        reference = dataset.at[idx, "output"] 
        generated_text = generate_response(model, tokenizer, instruction)

        bleu_score = sentence_bleu(reference, generated_text, weights=(0.25,0.25,0.25,0.25))
        bleu_scores.append(bleu_score)

        print(f"BLEU Score: {bleu_score}")
        print(f"Input: {instruction}")
        print(f"Reference: {reference}")
        print(f"Generated: {generated_text}\n")
                # DataFrame에 결과 추가
        results_df = pd.concat([results_df, pd.DataFrame({
            "bleu_score": [bleu_score],
            "instruction": [instruction],
            "reference": [reference],
            "generated": [generated_text]
        })], ignore_index=True)
        results_df.to_csv("llama-3.2-Korean-Bllossom-3B_evaluation.csv", index=False)
    # CSV 파일로 저장
    return bleu_scores

In [ ]:
if __name__ == "__main__":
    parquet_path = "train-00000-of-00001-1ae224438dce829b.parquet"  
    model_path = "./models--Bllossom--llama-3.2-Korean-Bllossom-3B/snapshots/67c476d8de2afd021661a40c0e5f5cbb51e62bc6"  
#models--Bllossom--llama-3.2-Korean-Bllossom-3B
    model = AutoModelForCausalLM.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    # 평가 실행
    evaluate_model(dataset, model, tokenizer)

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at ./models--Bllossom--llama-3.2-Korean-Bllossom-3B/snapshots/67c476d8de2afd021661a40c0e5f5cbb51e62bc6 and are newly initialized: ['model.layers.20.input_layernorm.weight', 'model.layers.20.mlp.down_proj.weight', 'model.layers.20.post_attention_layernorm.weight', 'model.layers.21.input_layernorm.weight', 'model.layers.21.mlp.down_proj.weight', 'model.layers.21.mlp.gate_proj.weight', 'model.layers.21.mlp.up_proj.weight', 'model.layers.21.post_attention_layernorm.weight', 'model.layers.21.self_attn.k_proj.weight', 'model.layers.21.self_attn.o_proj.weight', 'model.layers.21.self_attn.q_proj.weight', 'model.layers.21.self_attn.v_proj.weight', 'model.layers.22.input_layernorm.weight', 'model.layers.22.mlp.down_proj.weight', 'model.layers.22.mlp.gate_proj.weight', 'model.layers.22.mlp.up_proj.weight', 'model.layers.22.post_attention_layernorm.weight', 'model.layers.22.self_attn.k_proj.weight', 'model.layers.22.se